# Binary Classification Using Machine Learning
## Project Notebook: `Binary_Classification.ipynb`

**Objective:** Build and evaluate machine-learning models that classify observations into one of two categories.

**Dataset:** The built-in Breast Cancer Wisconsin dataset from `scikit-learn`. It contains numerical measurements computed from digitized images of breast-mass samples. The target labels are **malignant** and **benign**.

> **Educational note:** This is a machine-learning demonstration, not a medical diagnostic tool. Predictions must not be used to make real healthcare decisions.

### Project workflow
1. Import libraries and load the dataset.
2. Explore and prepare the data.
3. Split the data into training and testing sets.
4. Train Logistic Regression and Random Forest models.
5. Evaluate performance using accuracy, precision, recall, F1-score, confusion matrix, and ROC-AUC.
6. Compare models and inspect feature importance.

## 1. Import the required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score, RocCurveDisplay
)

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 2. Load the dataset

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

# Target mapping: 0 = malignant, 1 = benign
target_names = {0: "malignant", 1: "benign"}

print("Dataset shape:", X.shape)
print("\nTarget classes:")
print(pd.Series(y).map(target_names).value_counts())

display(X.head())

## 3. Explore the data

In [ ]:
print("Missing values in dataset:", int(X.isna().sum().sum()))
print("\nFeature summary:")
display(X.describe().T.head(10))

class_counts = y.map(target_names).value_counts()
class_counts.plot(kind="bar", rot=0)
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.show()

## 4. Split into training and testing sets

The test set is held out until evaluation. Stratification keeps the class proportions similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 5. Train the models

In [ ]:
# Logistic Regression benefits from feature scaling, so use a pipeline.
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
])

# Random Forest does not require feature scaling.
forest_model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

logistic_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

print("Both models trained successfully.")

## 6. Evaluate model performance

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    }

results = pd.DataFrame([
    evaluate_model(logistic_model, X_test, y_test, "Logistic Regression"),
    evaluate_model(forest_model, X_test, y_test, "Random Forest")
]).set_index("Model")

display(results.round(3))

## 7. Confusion matrix and classification report

In [ ]:
best_model_name = results["F1-score"].idxmax()
best_model = logistic_model if best_model_name == "Logistic Regression" else forest_model
y_pred = best_model.predict(X_test)

print("Model selected by F1-score:", best_model_name)
print("\nClassification report (0 = malignant, 1 = benign):")
print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["malignant", "benign"],
    cmap="Blues",
    values_format="d"
)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

## 8. Compare ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_estimator(logistic_model, X_test, y_test, name="Logistic Regression", ax=ax)
RocCurveDisplay.from_estimator(forest_model, X_test, y_test, name="Random Forest", ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
ax.set_title("ROC Curve Comparison")
plt.tight_layout()
plt.show()

## 9. Feature importance (Random Forest)

In [ ]:
importances = pd.Series(
    forest_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

display(importances.head(10).to_frame("Importance"))

importances.head(10).sort_values().plot(kind="barh", figsize=(8, 5))
plt.title("Top 10 Features — Random Forest")
plt.xlabel("Feature importance")
plt.tight_layout()
plt.show()

## 10. Try a sample prediction

The example below uses one row from the held-out test set. The displayed prediction is only for demonstrating the code and is not a medical conclusion.

In [ ]:
sample = X_test.iloc[[0]]
actual_label = int(y_test.iloc[0])
predicted_label = int(best_model.predict(sample)[0])
predicted_probability = float(best_model.predict_proba(sample)[0, 1])

print("Actual class:", target_names[actual_label])
print("Predicted class:", target_names[predicted_label])
print(f"Model probability for benign class: {predicted_probability:.3f}")

## 11. Conclusion

This project demonstrates a complete binary-classification workflow using two algorithms: Logistic Regression and Random Forest. The evaluation table compares accuracy, precision, recall, F1-score, and ROC-AUC. The confusion matrix shows the types of correct and incorrect predictions, while the ROC curve compares discrimination performance.

**Important:** Results depend on the selected dataset, train/test split, and model settings. This notebook is intended for learning machine learning concepts only.

### How to run
1. Install Python 3.10+ and Jupyter Notebook, or open the notebook in Google Colab.
2. Install dependencies with `pip install -r requirements.txt`.
3. Open `Binary_Classification.ipynb`.
4. Run the cells from top to bottom.